[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Files and Paths


## What you will be able to do

Read and write real files, build paths that work on any operating system, and handle the file
that is not there. You will also be able to say what the `with` statement does and why file code
is always written that way.


## The idea

### The problem

Everything you have worked on so far was typed into a cell. That is fine for learning and
useless for work, because real data lives in files: a spreadsheet export, a log, a list of
records someone sent you, the results your own program produced yesterday.

Two things have to happen for that to be useful. The program has to **find** the file, and it
has to **read** it without leaving anything in a bad state.

Neither is as simple as it sounds. Paths are written differently on different systems, so text
that works on one machine breaks on another. And a file that is opened has to be closed, even
when the code between raises an exception, or the operating system keeps the handle open and
the data you wrote may never reach the disk.

### What a path and a file object are

> A **path** identifies a location in the filesystem. `pathlib.Path` represents one as an object
> that knows how to join, split and inspect itself, rather than as a string you assemble by
> hand.
>
> Opening a path gives a **file object**: a connection to the file's contents, which must be
> closed when you are done. The `with` statement closes it for you, including when an exception
> is raised.

### Why with, rather than open and close

You can open a file, work with it and close it yourself. The problem is the middle:

```
f = open("data.txt")
rows = process(f)      # if this raises, close never runs
f.close()
```

`with` removes the possibility. It closes the file when the block ends, whether that is normal
completion or an exception, which is the `finally` behavior from the **Errors and Exceptions**
notebook applied automatically.

Write file code with `with`. There is no case in this guide where the manual form is better.

### Where you will meet this

The **Files, Paths and Formats** guide covers this properly: CSV, JSON, Excel, encodings,
directories, and writing safely so a failure part way through does not leave a half-written
file. The **Pandas** guide reads files by the thousand.

This notebook is the foundation both rest on, and enough on its own for reading a text file and
writing one back.

### A note on what this notebook creates

The cells below make a folder called `scratch` next to this notebook, write files into it, and
delete the whole thing at the end. In Colab you can watch it appear in the file browser on the
left. Nothing outside that folder is touched.

### What this notebook covers

- `pathlib.Path`, and joining paths with `/`
- The parts of a path: name, stem, suffix, parent
- `read_text` and `write_text`, for when the whole file fits in memory
- `open` with `with`, and reading a file one line at a time
- Writing and appending, and the mode that silently destroys a file
- Listing a folder, and matching names with `glob`
- Checking whether a file exists, and handling the case where it does not
- Four errors, one of which deletes your data without raising

Start by making somewhere to work.


In [1]:
from pathlib import Path

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print(scratch, "exists:", scratch.exists())


scratch exists: True


`exist_ok=True` means running the cell twice is not an error, which matters in a notebook where
cells get re-run constantly.


## Setup

Two imports from the standard library. `shutil` is used only by the cleanup cell at the end.


In [2]:
from pathlib import Path
import shutil

print("Ready.")


Ready.


## Worked examples

### Building a path

Join parts with `/`. It reads like a path and produces the right separator for the system the
code is running on.


In [3]:
note = scratch / "note.txt"

print(note)
print(type(note))


scratch/note.txt
<class 'pathlib.PosixPath'>


Writing `"scratch" + "/" + "note.txt"` happens to work on Linux and macOS and is wrong on
Windows, where the separator is a backslash. `Path` handles that, which is the main reason to
use it rather than strings.

A path can be built up in stages, and nothing touches the disk until you ask it to.


In [4]:
deeper = scratch / "reports" / "2026" / "march.csv"

print(deeper)
print("exists:", deeper.exists())


scratch/reports/2026/march.csv
exists: False


### The parts of a path


In [5]:
print("full:  ", deeper)
print("name:  ", deeper.name)
print("stem:  ", deeper.stem)
print("suffix:", deeper.suffix)
print("parent:", deeper.parent)


full:   scratch/reports/2026/march.csv
name:   march.csv
stem:   march
suffix: .csv
parent: scratch/reports/2026


`stem` and `suffix` are what you want when renaming a file or checking its type. Splitting on
`"."` by hand breaks on a name like `report.final.csv`, and `suffix` does not.


### Writing and reading the whole file

For anything that fits comfortably in memory, `write_text` and `read_text` are one line each.


In [6]:
note = scratch / "note.txt"

note.write_text("first line\nsecond line\nthird line\n")

print(note.read_text())


first line
second line
third line



`\n` is the newline from the **Strings** notebook, and it is what separates lines in a text
file.

`read_text` gives one string containing the whole file, newlines included.


In [7]:
contents = note.read_text()

print(repr(contents))
print(len(contents), "characters")


'first line\nsecond line\nthird line\n'
34 characters


### Reading line by line

For a large file, reading it all at once is wasteful. Opening it and looping gives one line at a
time, and only one is in memory at once.


In [8]:
with open(note) as f:
    for line in f:
        print(repr(line))


'first line\n'
'second line\n'
'third line\n'


Each line keeps its trailing newline, which is usually not what you want. `.rstrip()` from the
**Strings** notebook removes it.


In [9]:
with open(note) as f:
    for number, line in enumerate(f, start=1):
        print(number, line.rstrip())


1 first line
2 second line
3 third line


The `with` block is what closes the file. After the block ends, the file object is no longer
usable, which is the point.

`splitlines()` does the same job when you already have the text.


In [10]:
for line in note.read_text().splitlines():
    print(repr(line))


'first line'
'second line'
'third line'


### Writing, and the mode that destroys

`open` takes a mode. The three worth knowing are `"r"` to read, which is the default, `"w"` to
write, and `"a"` to append.


In [11]:
log = scratch / "log.txt"

with open(log, "w") as f:
    f.write("first entry\n")
    f.write("second entry\n")

print(log.read_text())


first entry
second entry



`"w"` **empties the file before writing.** Opening an existing file with `"w"` destroys its
contents immediately, before you write anything, and there is no warning and no error.


In [12]:
with open(log, "w") as f:
    f.write("this is now the whole file\n")

print(log.read_text())


this is now the whole file



Two entries gone. That is the mode working as designed, and it is the most common way people
lose data while learning this.

`"a"` appends instead, leaving what is there.


In [13]:
with open(log, "a") as f:
    f.write("added to the end\n")

print(log.read_text())


this is now the whole file
added to the end



Note that `write` does not add a newline. Every line you want has to end with `\n` explicitly,
which is the opposite of `print`.


### Looking at a folder

`iterdir` lists what is in a directory.


In [14]:
(scratch / "notes.md").write_text("# heading\n")
(scratch / "data.csv").write_text("a,b\n1,2\n")

for item in sorted(scratch.iterdir()):
    print(item.name)


data.csv
log.txt
note.txt
notes.md


`glob` matches names by pattern, which is usually what you want.


In [15]:
print("text files:", sorted(p.name for p in scratch.glob("*.txt")))
print("everything:", sorted(p.name for p in scratch.glob("*")))


text files: ['log.txt', 'note.txt']
everything: ['data.csv', 'log.txt', 'note.txt', 'notes.md']


`*` matches any run of characters. `glob("**/*.csv")` searches subfolders too, which the
**Files, Paths and Formats** guide covers along with the rest of `pathlib`.


### The file that is not there

This is the most common failure in file code, and it is not a bug. Files move, names are
mistyped, and a program that runs tomorrow may not find what was there today.


In [16]:
missing = scratch / "does_not_exist.txt"

print("exists:", missing.exists())


exists: False


`exists()` asks without raising. That is enough when you can do something sensible instead.


In [17]:
if missing.exists():
    print(missing.read_text())
else:
    print("no file yet, starting fresh")


no file yet, starting fresh


Catching the exception is the other approach, and it is better when the file is expected to be
there and its absence means something is wrong.


In [18]:
try:
    print(missing.read_text())
except FileNotFoundError as e:
    print("could not read it:", e)


could not read it: [Errno 2] No such file or directory: 'scratch/does_not_exist.txt'


Both are correct. Use `exists()` when a missing file is a normal case you handle. Use
`try` and `except FileNotFoundError` when it is a problem you want reported, which is the same
choice the **Errors and Exceptions** notebook described.


### Cleaning up

The folder and everything in it, removed.


In [19]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


`rmtree` deletes a directory and its contents with no confirmation and no undo. It is the right
tool here and worth respecting: point it at the wrong path and the files are gone.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Each task assumes a folder to work in, so start by running this:

```
from pathlib import Path
work = Path("practice")
work.mkdir(exist_ok=True)
```

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/17-files-and-paths-solutions.ipynb).

**1.** Create the `practice` folder, then write the text `"hello\nworld\n"` into
`practice/greeting.txt` and print it back.


In [20]:
# your code here


**2.** Print the `name`, `stem`, `suffix` and `parent` of `practice/greeting.txt`.


In [21]:
# your code here


**3.** Open `practice/greeting.txt` with `with` and print each line numbered, without the
trailing newline.


In [22]:
# your code here


**4.** Append a third line to the same file, then print the whole file to show all three lines.


In [23]:
# your code here


**5.** Write two more files into `practice`, one ending `.txt` and one ending `.csv`, then print
only the `.txt` filenames using `glob`.


In [24]:
# your code here


**6.** Try to read `practice/nothing.txt` and print `"not found"` instead of raising. Then
delete the `practice` folder with `shutil.rmtree`.


In [25]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### FileNotFoundError: the path does not exist


In [26]:
Path("no_such_file.txt").read_text()


FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.txt'

`[Errno 2] No such file or directory` names the path it tried. Check the spelling first, then
check where the program is actually running: a relative path like `data.txt` is looked up from
the current working directory, not from wherever you think the file lives.


In [27]:
import os

print("looking from:", os.getcwd())


looking from: /Users/johnfisher/Projects/Python-Visual-Guides/site/notebooks/python-from-the-start


### FileNotFoundError again: the parent folder does not exist

Writing a file does not create the folders above it.


In [28]:
Path("nowhere/deep/output.txt").write_text("data")


FileNotFoundError: [Errno 2] No such file or directory: 'nowhere/deep/output.txt'

The message looks the same as reading a missing file, and the cause is different: the file could
not be created because `nowhere/deep` does not exist.

`mkdir(parents=True, exist_ok=True)` makes the whole chain first.


In [29]:
target = Path("scratch2/deep/output.txt")
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text("data")

print(target.read_text())
shutil.rmtree("scratch2")


data


### TypeError: reading a file object twice

A file object is consumed as you read it.


In [30]:
tmp = Path("temp_demo.txt")
tmp.write_text("a\nb\n")

with open(tmp) as f:
    first = f.read()
    second = f.read()

print("first: ", repr(first))
print("second:", repr(second))


first:  'a\nb\n'
second: ''


The second read returned an empty string, not an error. The position is already at the end of
the file, so there is nothing left to read.

This is the same shape as the generator in the **Comprehensions** notebook: single pass, and the
second attempt quietly returns nothing. Read once into a variable, or reopen the file.


In [31]:
text = tmp.read_text()

print(repr(text))
print(repr(text))     # a string can be used any number of times


'a\nb\n'
'a\nb\n'


### The quiet one: opening with "w" to read


In [32]:
tmp.write_text("important data\n")
print("before:", repr(tmp.read_text()))

with open(tmp, "w") as f:
    pass                      # opened and closed, nothing written

print("after: ", repr(tmp.read_text()))
tmp.unlink()


before: 'important data\n'
after:  ''


The file is empty, and no error was raised at any point. The block did nothing at all; opening
with `"w"` truncated the file the moment it was opened.

This happens when `"w"` is typed instead of `"r"`, which is one keystroke. There is no undo. If
a file matters, the **Files, Paths and Formats** guide covers writing to a temporary file and
renaming it into place, which makes the operation safe.


## Recap

- `Path` builds paths with `/`, and works on every operating system.
- `name`, `stem`, `suffix` and `parent` split a path without string surgery.
- `read_text` and `write_text` handle a whole file in one line.
- `with open(...) as f:` closes the file even when the block raises.
- Looping over a file object gives one line at a time, with the newline still attached.
- `"w"` empties the file on open, `"a"` appends, and `write` adds no newline of its own.
- `iterdir` lists a folder and `glob` matches by pattern.
- Use `exists()` when a missing file is normal, and `except FileNotFoundError` when it is not.


## What is next

The **Modules and Imports** notebook, which moves code out of a single notebook and into files
you can import. It is what turns a set of cells into something you can reuse from anywhere.


---

&#8592; **Previous:** [Errors and Exceptions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/16-errors-and-exceptions.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)
